In [5]:
import pdfplumber
import re
from bs4 import BeautifulSoup
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [6]:
import sys

sys.path.append("..")

from preprocessing.text_cleaner import advanced_clean_text

from preprocessing.skill_extractor import advanced_skill_extractor

from pdf_parser.pdf_reader import extract_text_from_pdf

from matching.semantic_matcher import calculate_semantic_similarity

In [7]:
pdf_path = "../sample_resume/sample.pdf"

full_text = extract_text_from_pdf(pdf_path)

print(full_text[:3000])

INFORMATION TECHNOLOGY MANAGER
Summary
Dedicated IT Manager well-versed in analyzing and mitigating risk and finding cost-effective solutions. Excels at boosting performance and
productivity by establishing realistic goals and enforcing deadlines.
Highlights
Operations management
Salary structure/compensation analysis
Project trackingÂ
Calm under pressure
Performance criteria tracking
Compensation/benefits administration
Waterfall framework
Staff development
Scrum methodology
Client communication
Enterprise platforms
Experience
Information Technology Manager , 03/2013 to Current Company Name ï¼​ City , State
Managed a four-person local IT team, allocating resources to ongoing projects and enforcing deadlines.
Drove business KPIs through rapid iteration of customer-facing product features.
Leveraged in-depth understanding of end-to-end customer experience to identify pain points and latent customer needs.
Collaborated with the global team to resolve IT support cases.
Build and maintain 

In [8]:
cleaned_pdf_resume = advanced_clean_text(
    full_text
)

print(cleaned_pdf_resume[:2000])

information technology manager summary dedicated manager well versed analyzing mitigating risk finding cost effective solution excels boosting performance productivity establishing realistic goal enforcing deadline highlight operation management salary structure compensation analysis project tracking calm pressure performance criterion tracking compensation benefit administration waterfall framework staff development scrum methodology client communication enterprise platform experience information technology manager current company name city state managed four person local team allocating resource ongoing project enforcing deadline drove business kpis rapid iteration customer facing product feature leveraged depth understanding end end customer experience identify pain point latent customer need collaborated global team resolve support case build maintain staff five terminate cause one employee create audit process interlocking team adjust required manage travel budget staff site visit

In [9]:
pdf_resume_skills = advanced_skill_extractor(
    cleaned_pdf_resume
)

print("Extracted Skills:")

print(pdf_resume_skills)

Extracted Skills:
['communication', 'sql']


In [10]:
import pandas as pd

job_df = pd.read_csv(
    "../datasets/jobs.csv"
)

job_df.head()

,Unnamed: 0,Job Title,Job Description
0,0,Flutter Developer,We are looking for hire experts flutter develo...
1,1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n..."
3,3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside o...
4,4,Full Stack Developer,job responsibility full stack engineer – react...


In [11]:
job_df["cleaned_job_description"] = (
    job_df["Job Description"]
    .astype(str)
    .apply(advanced_clean_text)
)

In [12]:
sample_job = job_df[
    "cleaned_job_description"
][0]

semantic_score = calculate_semantic_similarity(
    cleaned_pdf_resume,
    sample_job
)

print("Semantic Match Score:")

print(f"{semantic_score}%")

Semantic Match Score:
38.40999984741211%


In [13]:
all_jobs = job_df[
    "cleaned_job_description"
].tolist()

job_titles = job_df[
    "Job Title"
].tolist()

In [14]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [15]:
job_embeddings = model.encode(
    all_jobs
)

In [16]:
resume_embedding = model.encode(
    cleaned_pdf_resume
)

In [17]:
from sklearn.metrics.pairwise import cosine_similarity

all_scores = cosine_similarity(
    [resume_embedding],
    job_embeddings
)

In [18]:
scores = all_scores[0]

In [19]:
results_df = pd.DataFrame({

    "Job Title": job_titles,

    "Match Score": scores
})

In [20]:
results_df["Match Score"] = (
    results_df["Match Score"] * 100
).round(2)

In [21]:
top_jobs = results_df.sort_values(
    by="Match Score",
    ascending=False
)

In [22]:
top_jobs.head(10)

,Job Title,Match Score
1919,Database Administrator,81.389999
1655,Network Administrator,80.610001
1951,Software Engineer,80.400002
573,Java Developer,78.970001
1511,DevOps Engineer,78.690002
947,Java Developer,78.370003
942,iOS Developer,78.260002
1271,Database Administrator,77.940002
18,Database Administrator,77.440002
1141,Node js developer,77.419998
